# HTTPX

httpx is a Python library, the successor to requests, that enables async calls built on top of asyncio (Python's async/await library).

## Install

\> poetry add httpx

## Usage

### Sync requests

In [62]:
import httpx

rget = httpx.get('https://httpbin.org/get')
rgetparams = httpx.get('https://httpbin.org/get', params={'key1': 'value1', 'key2': 'value2'}, cookies={'peanut': 'butter'})
rpost = httpx.post('https://httpbin.org/post', data={'key': 'value'})
rput = httpx.put('https://httpbin.org/put', data={'key': 'value'})
rdel = httpx.delete('https://httpbin.org/delete')

print(rget)
print(rget.status_code)
print(rget.text)
print(rgetparams)

<Response [200 OK]>
200
{
  "args": {}, 
  "headers": {
    "Accept": "*/*", 
    "Accept-Encoding": "gzip, deflate", 
    "Host": "httpbin.org", 
    "User-Agent": "python-httpx/0.28.1", 
    "X-Amzn-Trace-Id": "Root=1-6a85dfc1-059b680a690b30f742aa4637"
  }, 
  "origin": "79.6.168.117", 
  "url": "https://httpbin.org/get"
}

<Response [200 OK]>


### Sync streaming

when using stream response.content and response.text are not available.

In [66]:
with httpx.stream("GET", "https://www.example.com") as r:
    # for data in r.iter_bytes():
    # for line in r.iter_lines():
    # for chunk in r.iter_raw():
    for text in r.iter_text(chunk_size=128):
        print(f'{text}\n')



<!doctype html><html lang="en"><head><title>Example Domain</title><link rel="icon" href="data:,"><meta name="viewport" content="

width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1

{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is 

for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/e

xample">Learn more</a></p></div></body></html>




### Raise errors on 4xx and 5xx

In [22]:
try:
    r = httpx.get('https://httpbin.org/getwrong') # this will return a 404 response
    r.raise_for_status() # throws an exception if the response was an error (4xx or 5xx)
except httpx.HTTPStatusError as exc:
    print(f'Error response {exc.response.status_code} while requesting {exc.request.url!r}.')

DEBUG [2026-08-20 18:57:17] httpcore.connection - connect_tcp.started host='httpbin.org' port=443 local_address=None timeout=5.0 socket_options=None
DEBUG [2026-08-20 18:57:17] httpcore.connection - connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001BABDF3FE00>
DEBUG [2026-08-20 18:57:17] httpcore.connection - start_tls.started ssl_context=<ssl.SSLContext object at 0x000001BABDE01E50> server_hostname='httpbin.org' timeout=5.0
DEBUG [2026-08-20 18:57:18] httpcore.connection - start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000001BABF86E210>
DEBUG [2026-08-20 18:57:18] httpcore.http11 - send_request_headers.started request=<Request [b'GET']>
DEBUG [2026-08-20 18:57:18] httpcore.http11 - send_request_headers.complete
DEBUG [2026-08-20 18:57:18] httpcore.http11 - send_request_body.started request=<Request [b'GET']>
DEBUG [2026-08-20 18:57:18] httpcore.http11 - send_request_body.complete
DEBUG [2026-08-20 18:57:18] httpcor

Error response 404 while requesting URL('https://httpbin.org/getwrong').


### Client

When you make requests directly using httpx you create a new connection for every single request.

In production you must use Client because it implements connection pooling, reusing the underlying TCP connection across requests to the same host.

Headers, params and cookies can be specified both at client level and request level and they will be combined. For all other parameters, the request-level values take priority.

In [64]:
with httpx.Client(headers={'user-agent': 'my-app/0.0.1'},
                  params = {'client_id': 'client1'},
                  base_url='https://httpbin.org',
                  auth=('name', 'george')) as client:
    rget = client.get('/get', headers={'x-custom': 'from-request'})
    rget = client.get('/get', headers={'x-custom': 'from-request'})
    rget = client.get('/get', headers={'x-custom': 'from-request'})
    print(f'response: {rget}')
    print(f'request url: {rget.request.url}')
    print(f'request headers: {rget.request.headers}')

response: <Response [200 OK]>
request url: https://httpbin.org/get?client_id=client1
request headers: Headers({'host': 'httpbin.org', 'accept': '*/*', 'accept-encoding': 'gzip, deflate', 'connection': 'keep-alive', 'user-agent': 'my-app/0.0.1', 'x-custom': 'from-request', 'authorization': '[secure]'})


### Async client

Async client lets you make async requests.

A single AsyncClient instance can be used to spawn multiple concurrent requests at once.

In [ ]:
import asyncio

async with httpx.AsyncClient() as client:
    rget = await client.get('https://example.com')
    rget = await client.get('https://example.com')
    rget = await client.get('https://example.com')
    print(rget)
# this takes 0.42 seconds (3 requests in sequence)

urls = ["https://example.com"] * 10
async with httpx.AsyncClient() as client:
    results = await asyncio.gather(*(client.get(u) for u in urls))
    print(results)
# this takes 0.36 seconds (10 requests in parallel)


async with httpx.AsyncClient() as client:
    tasks = []
    for u in urls:
        tasks.append(asyncio.create_task(client.get(u))) # using create_task the coroutines will start immediately, instead of waiting for the gather to start them
    # do something heavy
    #
    #
    #
    results = await asyncio.gather(*tasks)
    print(results)

<Response [200 OK]>
[<Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>]
[<Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>, <Response [200 OK]>]


you can also async streaming

In [69]:
client = httpx.AsyncClient()
async with client.stream('GET', 'https://www.example.com/') as response:
    async for chunk in response.aiter_bytes(chunk_size=128):
        print(f'{chunk}\n')
    # response.aiter_text()
    # response.aiter_lines()
    # response.aiter_raw()

b'<!doctype html><html lang="en"><head><title>Example Domain</title><link rel="icon" href="data:,"><meta name="viewport" content="'

b'width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1'

b'{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is '

b'for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/e'

b'xample">Learn more</a></p></div></body></html>\n'



### Timeout

Default timeout in httpx is 5 seconds, which is way too short for LLMs

In [3]:
async with httpx.AsyncClient(timeout=60.0) as client:
    r = await client.get('https://example.com')
print(r)

DEBUG [2026-08-20 18:47:01] httpcore.connection - connect_tcp.started host='example.com' port=443 local_address=None timeout=60.0 socket_options=None
DEBUG [2026-08-20 18:47:01] httpcore.connection - connect_tcp.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001BABD517610>
DEBUG [2026-08-20 18:47:01] httpcore.connection - start_tls.started ssl_context=<ssl.SSLContext object at 0x000001BABD446210> server_hostname='example.com' timeout=60.0
DEBUG [2026-08-20 18:47:01] httpcore.connection - start_tls.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001BABD487950>
DEBUG [2026-08-20 18:47:01] httpcore.http11 - send_request_headers.started request=<Request [b'GET']>
DEBUG [2026-08-20 18:47:01] httpcore.http11 - send_request_headers.complete
DEBUG [2026-08-20 18:47:01] httpcore.http11 - send_request_body.started request=<Request [b'GET']>
DEBUG [2026-08-20 18:47:01] httpcore.http11 - send_request_body.complete
DEBUG [2026-08-20 18:47:01] h

<Response [200 OK]>


### Logging

In [ ]:
import logging

# this will debug level to stndout
logging.basicConfig(
    format="%(levelname)s [%(asctime)s] %(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.DEBUG,
    force=True # used to override ipynb config
)

#here you can make the request

### Retries without tenacity

Note: this is just a sample to show how to handle requests in a robust way and so it's not meant to be generic. we'll use tenacity anyway, so we don't need to make a more complex example.

In [83]:
import httpx
import random
import asyncio
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

async def robust_request(client: httpx.AsyncClient, url: str, max_retries: int, base: float = 1, max_wait: float = 30):
        attempt = 0
        r = httpx.Response(400)

        while(r.is_error and attempt < max_retries):
            attempt += 1
            try:
                r = await client.get(url)
                r.raise_for_status()
            # HTTPError also catch 
            except httpx.HTTPError as e:
                logging.log(logging.ERROR, f"error {e!r} - attempt: {attempt}")
                if attempt >= max_retries:
                    break

                #exp backoff with full jitter
                wait = min(base * (2**attempt), max_wait)
                wait = random.uniform(0, wait)

                #overwrite wait if error is 429 and retry-after has been sent by the server
                if isinstance(e, httpx.HTTPStatusError) and r.status_code == 429:
                    retry_after = parse_retry_after(r.headers.get("Retry-After"))
                    wait = retry_after if retry_after is not None else wait

                await asyncio.sleep(wait)

        return r

def parse_retry_after(retry_after: str) -> float:
    if not retry_after:
        return None
    # number of seconds
    if retry_after.isdigit():
        return float(retry_after)
    # exact date
    try:
        retry_date = parsedate_to_datetime(retry_after)
        if retry_date.tzinfo is None:
            retry_date = retry_date.replace(tzinfo=timezone.utc)
        delta = (retry_date - datetime.now(timezone.utc)).total_seconds()
        return max(delta, 0)
    except (TypeError, ValueError):
            return None

async with httpx.AsyncClient() as client: 
    r = await robust_request(client, "https://httpbin.org/get", 3)



ERROR [2026-08-21 15:32:38] root - error ReadTimeout('') - attempt: 1
ERROR [2026-08-21 15:32:45] root - error ReadTimeout('') - attempt: 2


### Retries with tenacity

Install

\> poetry add tenacity

Usage

In [121]:
from tenacity import before_sleep_log, retry, stop_after_attempt, wait_exponential, retry_if_exception_type, RetryError
import logging

logger = logging.getLogger(__name__)

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type((httpx.TransportError, httpx.HTTPStatusError)),
    before_sleep=before_sleep_log(logger, logging.INFO),
    # reraise=True, # by default this is False and tenacity throws a RetryError if attempts are exhausted, set this to True in order to reraise the original exception
)
def unreliable_fetch(url):
    response = httpx.get(url)
    response.raise_for_status()
    return response

try:
    unreliable_fetch("https://httpbin.org/get")
except RetryError as exc:
    logger.log(logging.ERROR, f"Error - {exc!r}")

INFO [2026-08-21 17:21:45] __main__ - Retrying __main__.unreliable_fetch in 2 seconds as it raised ConnectError: [Errno 11001] getaddrinfo failed.
INFO [2026-08-21 17:21:48] __main__ - Retrying __main__.unreliable_fetch in 2 seconds as it raised ConnectError: [Errno 11001] getaddrinfo failed.
ERROR [2026-08-21 17:21:50] __main__ - Error - RetryError(<Future at 0x1bac2ca7850 state=finished raised ConnectError>)


Remember that retries should be used only on idempotent requests, otherwise you risk creating duplicates. You should decorate the function that wraps the unreliable request.

### Tips for usage with LLMs

* The default httpx timeout is 5 seconds, which is way too short for LLMs
* You can use tenacity (a library like Polly) to handle retries
* Many LLM SDKS already have an AsyncClient that wraps httpx, so you don't have to make the HTTP request directly